In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# mimic_data_preparation.py
import pandas as pd
import os
import re
from sklearn.model_selection import train_test_split

def preprocess_admissions(admissions_path):
    """Load and preprocess admissions data with only necessary columns."""
    admissions = pd.read_csv(
        admissions_path,
        usecols=['HADM_ID', 'HOSPITAL_EXPIRE_FLAG'],
        dtype={'HADM_ID': 'int32', 'HOSPITAL_EXPIRE_FLAG': 'int8'}
    )
    return admissions

def prepare_ftl_trans_data(admissions_path, notes_path, data_dir='/content/drive/My Drive/mimic/data', chunk_size=100000):
    """Prepare data by processing notes in chunks and saving splits to disk."""
    # Load and preprocess admissions
    admissions = preprocess_admissions(admissions_path)

    # Create a dictionary for quick label lookup
    label_dict = admissions.set_index('HADM_ID')['HOSPITAL_EXPIRE_FLAG'].to_dict()

    # Split unique HADM_IDs into train, val, test to avoid data leakage
    unique_adm = admissions['HADM_ID'].unique()
    train_adm, temp_adm = train_test_split(unique_adm, test_size=0.2, random_state=42)
    val_adm, test_adm = train_test_split(temp_adm, test_size=0.5, random_state=42)

    # Convert to sets for faster lookup
    train_adm_set = set(train_adm)
    val_adm_set = set(val_adm)
    test_adm_set = set(test_adm)

    # Create data directory if it doesn’t exist
    os.makedirs(data_dir, exist_ok=True)

    # Initialize CSV files with headers
    header = ['Adm_ID', 'Note_ID', 'chartdate', 'charttime', 'TEXT', 'Label']
    pd.DataFrame(columns=header).to_csv(os.path.join(data_dir, 'train.csv'), index=False)
    pd.DataFrame(columns=header).to_csv(os.path.join(data_dir, 'val.csv'), index=False)
    pd.DataFrame(columns=header).to_csv(os.path.join(data_dir, 'test.csv'), index=False)

    # Process notes in chunks
    for chunk in pd.read_csv(
        notes_path,
        chunksize=chunk_size,
        usecols=['HADM_ID', 'ROW_ID', 'CHARTDATE', 'CHARTTIME', 'TEXT'],
        dtype={'HADM_ID': 'Int32', 'ROW_ID': 'Int32'},
        parse_dates=['CHARTDATE', 'CHARTTIME']
    ):
        # Drop rows where HADM_ID is NA
        chunk = chunk.dropna(subset=['HADM_ID'])

        # Preprocess text with vectorized operations
        chunk['TEXT'] = chunk['TEXT'].str.replace(r'\[\*\*.*?\*\*\]', '', regex=True)
        chunk['TEXT'] = chunk['TEXT'].str.replace(r'[^\w\s]', '', regex=True).str.lower()

        # Impute missing CHARTTIME efficiently
        chunk['CHARTTIME'] = chunk['CHARTTIME'].where(
            chunk['CHARTTIME'].notna(),
            chunk['CHARTDATE'] + pd.Timedelta(hours=23, minutes=59, seconds=59)
        )

        # Add labels using the dictionary
        chunk['Label'] = chunk['HADM_ID'].map(label_dict)

        # Rename columns
        chunk = chunk.rename(columns={
            'HADM_ID': 'Adm_ID',
            'ROW_ID': 'Note_ID',
            'CHARTDATE': 'chartdate',
            'CHARTTIME': 'charttime',
            'TEXT': 'TEXT'
        })

        # Select required columns
        chunk = chunk[['Adm_ID', 'Note_ID', 'chartdate', 'charttime', 'TEXT', 'Label']]

        # Split the chunk into train, val, test
        train_chunk = chunk[chunk['Adm_ID'].isin(train_adm_set)]
        val_chunk = chunk[chunk['Adm_ID'].isin(val_adm_set)]
        test_chunk = chunk[chunk['Adm_ID'].isin(test_adm_set)]

        # Append to CSV files without rewriting headers
        train_chunk.to_csv(os.path.join(data_dir, 'train.csv'), mode='a', header=False, index=False)
        val_chunk.to_csv(os.path.join(data_dir, 'val.csv'), mode='a', header=False, index=False)
        test_chunk.to_csv(os.path.join(data_dir, 'test.csv'), mode='a', header=False, index=False)

    print(f"Data has been successfully saved to the '{data_dir}/' directory with train.csv, val.csv, and test.csv.")

if __name__ == "__main__":
    admissions_path = '/content/drive/My Drive/mimic/ADMISSIONS.csv'
    notes_path = '/content/drive/My Drive/mimic/NOTEEVENTS.csv'
    prepare_ftl_trans_data(admissions_path, notes_path)

Data has been successfully saved to the '/content/drive/My Drive/mimic/data/' directory with train.csv, val.csv, and test.csv.


In [ ]:
import pandas as pd
import os

def preprocess_admissions(admissions_path):
    """Load and preprocess admissions data with only necessary columns."""
    admissions = pd.read_csv(
        admissions_path,
        usecols=['HADM_ID', 'HOSPITAL_EXPIRE_FLAG'],
        dtype={'HADM_ID': 'int32', 'HOSPITAL_EXPIRE_FLAG': 'int8'}
    )
    return admissions

def prepare_ftl_trans_data(admissions_path, notes_path, data_dir='/content/drive/My Drive/mimic/data', chunk_size=100000):
    """Prepare data by processing notes in chunks and saving to a single TSV file."""
    # Load and preprocess admissions
    admissions = preprocess_admissions(admissions_path)

    # Create a dictionary for quick label lookup
    label_dict = admissions.set_index('HADM_ID')['HOSPITAL_EXPIRE_FLAG'].to_dict()

    # Create data directory if it doesn’t exist
    os.makedirs(data_dir, exist_ok=True)

    # Define the output TSV file path
    output_file = os.path.join(data_dir, 'original_data.tsv')

    # Flag to indicate if it's the first chunk
    first_chunk = True

    # Process notes in chunks
    for chunk in pd.read_csv(
        notes_path,
        chunksize=chunk_size,
        usecols=['HADM_ID', 'ROW_ID', 'CHARTDATE', 'CHARTTIME', 'TEXT'],
        dtype={'HADM_ID': 'Int32', 'ROW_ID': 'Int32'},
        parse_dates=['CHARTDATE', 'CHARTTIME']
    ):
        # Drop rows where HADM_ID is NA
        chunk = chunk.dropna(subset=['HADM_ID'])

        # Preprocess text with vectorized operations
        chunk['TEXT'] = chunk['TEXT'].str.replace(r'\[\*\*.*?\*\*\]', '', regex=True)
        chunk['TEXT'] = chunk['TEXT'].str.replace(r'[^\w\s]', '', regex=True).str.lower()

        # Impute missing CHARTTIME efficiently
        chunk['CHARTTIME'] = chunk['CHARTTIME'].where(
            chunk['CHARTTIME'].notna(),
            chunk['CHARTDATE'] + pd.Timedelta(hours=23, minutes=59, seconds=59)
        )

        # Add labels using the dictionary
        chunk['Label'] = chunk['HADM_ID'].map(label_dict)

        # Drop rows where Label is NaN (HADM_ID not in admissions)
        chunk = chunk.dropna(subset=['Label'])

        # Rename columns
        chunk = chunk.rename(columns={
            'HADM_ID': 'Adm_ID',
            'ROW_ID': 'Note_ID',
            'CHARTDATE': 'chartdate',
            'CHARTTIME': 'charttime',
            'TEXT': 'TEXT'
        })

        # Select required columns in order
        chunk = chunk[['Adm_ID', 'Note_ID', 'chartdate', 'charttime', 'TEXT', 'Label']]

        # Append to TSV file
        mode = 'w' if first_chunk else 'a'
        chunk.to_csv(output_file, sep='\t', header=False, index=False, mode=mode)

        # After the first chunk, set flag to False
        first_chunk = False

    print(f"Data has been successfully saved to '{output_file}' as a single TSV file.")

if __name__ == "__main__":
    admissions_path = '/content/drive/My Drive/mimic/ADMISSIONS.csv'
    notes_path = '/content/drive/My Drive/mimic/NOTEEVENTS.csv'
    prepare_ftl_trans_data(admissions_path, notes_path)

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install dependencies
!pip install pytorch-transformers==1.2.0

# Define and run
import subprocess

args = [
    'python', '/content/drive/My Drive/scripts/preprocessing.py',
    '--original_data', '/content/drive/My Drive/mimic/data/original_data.tsv',
    '--output_dir', '/content/drive/My Drive/mimic/data/processed',
    '--temp_dir', '/content/drive/My Drive/mimic/data/temp',
    '--task_name', 'my_task',
    '--log_path', '/content/drive/My Drive/mimic/data/log.txt',
    '--id_num_neg', '1000',
    '--id_num_pos', '1000',
    '--random_seed', '42',
    '--bert_model', 'bert-base-uncased'
]

result = subprocess.run(args, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

In [ ]:
# utils.py
from __future__ import print_function
from __future__ import absolute_import
from __future__ import division

import numpy as np
import six


def time_batch_generator(max_len, input_ids, labels, masks, note_ids, chunk_ids, times=None):
    """batch generator with note_id, chunk_id and time
    """
    size = len(input_ids)
    indices = np.arange(size)
    np.random.shuffle(indices)

    i = 0
    while True:
        if i < size:
            if times is not None:
                yield input_ids[indices[i]][-max_len:, :], labels[indices[i]], masks[indices[i]][-max_len:, :], \
                      note_ids[indices[i]][-max_len:], chunk_ids[indices[i]][-max_len:], times[indices[i]][-max_len:]
            else:
                yield input_ids[indices[i]][-max_len:, :], labels[indices[i]], masks[indices[i]][-max_len:, :], \
                      note_ids[indices[i]][-max_len:], chunk_ids[indices[i]][-max_len:]
            i += 1
        else:
            i = 0
            indices = np.arange(size)
            np.random.shuffle(indices)
            continue


def mask_batch_generator(max_len, input_ids, labels, masks):
    """batch generator
    """
    size = len(input_ids)
    indices = np.arange(size)
    np.random.shuffle(indices)

    i = 0
    while True:
        if i < size:
            yield input_ids[indices[i]][-max_len:, :], labels[indices[i]], masks[indices[i]][-max_len:, :]
            i += 1
        else:
            i = 0
            indices = np.arange(size)
            np.random.shuffle(indices)
            continue

def pad_sequences(sequences, maxlen=None, dtype='int32',
                  padding='pre', truncating='pre', value=0.):
    """Pads sequences to the same length.

    This function transforms a list of
    `num_samples` sequences (lists of integers)
    into a 2D Numpy array of shape `(num_samples, num_timesteps)`.
    `num_timesteps` is either the `maxlen` argument if provided,
    or the length of the longest sequence otherwise.

    Sequences that are shorter than `num_timesteps`
    are padded with `value` at the end.

    Sequences longer than `num_timesteps` are truncated
    so that they fit the desired length.
    The position where padding or truncation happens is determined by
    the arguments `padding` and `truncating`, respectively.

    Pre-padding is the default.

    # Arguments
        sequences: List of lists, where each element is a sequence.
        maxlen: Int, maximum length of all sequences.
        dtype: Type of the output sequences.
            To pad sequences with variable length strings, you can use `object`.
        padding: String, 'pre' or 'post':
            pad either before or after each sequence.
        truncating: String, 'pre' or 'post':
            remove values from sequences larger than
            `maxlen`, either at the beginning or at the end of the sequences.
        value: Float or String, padding value.

    # Returns
        x: Numpy array with shape `(len(sequences), maxlen)`

    # Raises
        ValueError: In case of invalid values for `truncating` or `padding`,
            or in case of invalid shape for a `sequences` entry.
    """
    if not hasattr(sequences, '__len__'):
        raise ValueError('`sequences` must be iterable.')
    num_samples = len(sequences)

    lengths = []
    for x in sequences:
        try:
            lengths.append(len(x))
        except TypeError:
            raise ValueError('`sequences` must be a list of iterables. '
                             'Found non-iterable: ' + str(x))

    if maxlen is None:
        maxlen = np.max(lengths)

    # take the sample shape from the first non empty sequence
    # checking for consistency in the main loop below.
    sample_shape = tuple()
    for s in sequences:
        if len(s) > 0:
            sample_shape = np.asarray(s).shape[1:]
            break

    is_dtype_str = np.issubdtype(dtype, np.str_) or np.issubdtype(dtype, np.unicode_)
    if isinstance(value, six.string_types) and dtype != object and not is_dtype_str:
        raise ValueError("`dtype` {} is not compatible with `value`'s type: {}\n"
                         "You should set `dtype=object` for variable length strings."
                         .format(dtype, type(value)))

    x = np.full((num_samples, maxlen) + sample_shape, value, dtype=dtype)
    for idx, s in enumerate(sequences):
        if not len(s):
            continue  # empty list/array was found
        if truncating == 'pre':
            trunc = s[-maxlen:]
        elif truncating == 'post':
            trunc = s[:maxlen]
        else:
            raise ValueError('Truncating type "%s" '
                             'not understood' % truncating)

        # check `trunc` has expected shape
        trunc = np.asarray(trunc, dtype=dtype)
        if trunc.shape[1:] != sample_shape:
            raise ValueError('Shape of sample %s of sequence at position %s '
                             'is different from expected shape %s' %
                             (trunc.shape[1:], idx, sample_shape))

        if padding == 'post':
            x[idx, :len(trunc)] = trunc
        elif padding == 'pre':
            x[idx, -len(trunc):] = trunc
        else:
            raise ValueError('Padding type "%s" not understood' % padding)
    return x

In [ ]:
# other_func.py
import os
import time
import re
import io
import torch
import numpy as np
import pandas as pd
from tqdm import trange, tqdm
from sklearn.metrics import roc_curve, precision_recall_curve, \
    auc, matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score

def write_log(content, log_path, print_content=True):
    if os.path.exists(log_path):
        with open(log_path, 'a') as f:
            f.write("Time: " + time.ctime() + "\n")
            f.write(content + "\n")
            f.write("=====================\n")
    else:
        with open(log_path, 'w') as f:
            f.write("Time: " + time.ctime() + "\n")
            f.write(content + "\n")
            f.write("=====================\n")
    if print_content:
        print(content)


def preprocess1(x):
    y = re.sub('\\[(.*?)\\]', '', x)  # remove de-identified brackets
    y = re.sub('[0-9]+\.', '', y)  # remove 1.2. since the segmenter segments based on this
    y = re.sub('dr\.', 'doctor', y)
    y = re.sub('m\.d\.', 'md', y)
    y = re.sub('admission date:', '', y)
    y = re.sub('discharge date:', '', y)
    y = re.sub('--|__|==', '', y)
    return y


def preprocessing(df_less_n, tokenizer):
    df_less_n['TEXT'] = df_less_n['TEXT'].fillna(' ')
    df_less_n['TEXT'] = df_less_n['TEXT'].str.replace('\n', ' ')
    df_less_n['TEXT'] = df_less_n['TEXT'].str.replace('\r', ' ')
    df_less_n['TEXT'] = df_less_n['TEXT'].apply(str.strip)
    df_less_n['TEXT'] = df_less_n['TEXT'].str.lower()

    df_less_n['TEXT'] = df_less_n['TEXT'].apply(lambda x: preprocess1(x))

    sen = df_less_n['TEXT'].values
    tokenized_texts = [tokenizer.tokenize(x) for x in sen]
    print("First sentence tokenized")
    print(tokenized_texts[0])
    input_ids = [tokenizer.convert_tokens_to_ids(x) for x in tokenized_texts]
    df_less_n['Input_ID'] = input_ids
    return df_less_n[['Adm_ID', 'Note_ID', 'TEXT', 'Input_ID', 'Label', 'chartdate', 'charttime']]


def word_count_pre(df_less_n):
    df_less_n['TEXT'] = df_less_n['TEXT'].fillna(' ')
    df_less_n['TEXT'] = df_less_n['TEXT'].str.replace('\n', ' ')
    df_less_n['TEXT'] = df_less_n['TEXT'].str.replace('\r', ' ')
    df_less_n['TEXT'] = df_less_n['TEXT'].apply(str.strip)
    df_less_n['TEXT'] = df_less_n['TEXT'].str.lower()
    df_less_n['TEXT'] = df_less_n['TEXT'].apply(lambda x: preprocess1(x))
    return df_less_n


def split_into_chunks(df, max_len):
    input_ids = df.Input_ID.apply(lambda x: x[1:-1].replace(' ', '').split(','))
    df_len = len(df)
    Adm_ID, Note_ID, Input_ID, Label, chartdate, charttime = [], [], [], [], [], []
    for i in tqdm(range(df_len)):
        x = input_ids[i]
        n = int(len(x) / (max_len - 2))
        for j in range(n):
            Adm_ID.append(df.Adm_ID[i])
            Note_ID.append(df.Note_ID[i])
            sub_ids = x[j * (max_len - 2): (j + 1) * (max_len - 2)]
            sub_ids.insert(0, '101')
            sub_ids.append('102')
            Input_ID.append(' '.join(sub_ids))
            Label.append(df.Label[i])
            chartdate.append(df.chartdate[i])
            charttime.append(df.charttime[i])
        if len(x) % (max_len - 2) > 10:
            Adm_ID.append(df.Adm_ID[i])
            Note_ID.append(df.Note_ID[i])
            sub_ids = x[-((len(x)) % (max_len - 2)):]
            sub_ids.insert(0, '101')
            sub_ids.append('102')
            Input_ID.append(' '.join(sub_ids))
            Label.append(df.Label[i])
            chartdate.append(df.chartdate[i])
            charttime.append(df.charttime[i])
    new_df = pd.DataFrame({'Adm_ID': Adm_ID,
                           'Note_ID': Note_ID,
                           'Input_ID': Input_ID,
                           'Label': Label,
                           'chartdate': chartdate,
                           'charttime': charttime})
    new_df = new_df.astype({'Adm_ID': 'int64', 'Note_ID': 'int64', 'Label': 'int64'})
    return new_df


def Tokenize(df, max_length, tokenizer):
    labels = df.Label.values
    if 'TEXT' in df.columns:
        sen = df.TEXT.values
        labels = df.Label.values
        sen = ["[CLS] " + x + " [SEP]" for x in sen]
        tokenized_texts = [tokenizer.tokenize(x) for x in sen]
        print("First sentence tokenized")
        print(tokenized_texts[0])
        input_ids = [tokenizer.convert_tokens_to_ids(x) for x in tokenized_texts]
    else:
        assert 'Input_ID' in df.columns
        input_ids = df.Input_ID.apply(lambda x: x.split(' '))
        input_ids = input_ids.apply(lambda x: [int(i) for i in x])
        input_ids = input_ids.values
    input_ids = pad_sequences(input_ids, maxlen=max_length, dtype="long", truncating="post", padding="post")
    attention_masks = []
    for seq in input_ids:
        seq_mask = [float(i > 0) for i in seq]
        attention_masks.append(seq_mask)
    return labels, input_ids, attention_masks


def Tokenize_with_note_id(df, max_length, tokenizer):
    labels = df.Label.values
    note_ids = df.Note_ID.values
    if 'TEXT' in df.columns:
        sen = df.TEXT.values
        labels = df.Label.values
        sen = ["[CLS] " + x + " [SEP]" for x in sen]
        tokenized_texts = [tokenizer.tokenize(x) for x in sen]
        print("First sentence tokenized")
        print(tokenized_texts[0])
        input_ids = [tokenizer.convert_tokens_to_ids(x) for x in tokenized_texts]
    else:
        assert 'Input_ID' in df.columns
        input_ids = df.Input_ID.apply(lambda x: x.split(' '))
        input_ids = input_ids.apply(lambda x: [int(i) for i in x])
        input_ids = input_ids.values
    input_ids = pad_sequences(input_ids, maxlen=max_length, dtype="long", truncating="post", padding="post")
    attention_masks = []
    for seq in input_ids:
        seq_mask = [float(i > 0) for i in seq]
        attention_masks.append(seq_mask)
    return labels, input_ids, attention_masks, note_ids


def Tokenize_with_note_id_time(df, max_length, tokenizer):
    labels = df.Label.values
    note_ids = df.Note_ID.values
    times = pd.to_datetime(df.chartdate.values)
    times = times - times.min()
    times = times.days.values
    if 'TEXT' in df.columns:
        sen = df.TEXT.values
        labels = df.Label.values
        sen = ["[CLS] " + x + " [SEP]" for x in sen]
        tokenized_texts = [tokenizer.tokenize(x) for x in sen]
        print("First sentence tokenized")
        print(tokenized_texts[0])
        input_ids = [tokenizer.convert_tokens_to_ids(x) for x in tokenized_texts]
    else:
        assert 'Input_ID' in df.columns
        input_ids = df.Input_ID.apply(lambda x: x.split(' '))
        input_ids = input_ids.apply(lambda x: [int(i) for i in x])
        input_ids = input_ids.values
    input_ids = pad_sequences(input_ids, maxlen=max_length, dtype="long", truncating="post", padding="post")
    attention_masks = []
    for seq in input_ids:
        seq_mask = [float(i > 0) for i in seq]
        attention_masks.append(seq_mask)
    return labels, input_ids, attention_masks, note_ids, times


def Tokenize_with_note_id_hour(df, max_length, tokenizer):
    labels = df.Label.values
    note_ids = df.Note_ID.values
    times = pd.to_datetime(df.charttime.values)
    times = times - times.min()
    times = times / pd.Timedelta(days=1)
    if 'TEXT' in df.columns:
        sen = df.TEXT.values
        labels = df.Label.values
        sen = ["[CLS] " + x + " [SEP]" for x in sen]
        tokenized_texts = [tokenizer.tokenize(x) for x in sen]
        print("First sentence tokenized")
        print(tokenized_texts[0])
        input_ids = [tokenizer.convert_tokens_to_ids(x) for x in tokenized_texts]
    else:
        assert 'Input_ID' in df.columns
        input_ids = df.Input_ID.apply(lambda x: x.split(' '))
        input_ids = input_ids.apply(lambda x: [int(i) for i in x])
        input_ids = input_ids.values
    input_ids = pad_sequences(input_ids, maxlen=max_length, dtype="long", truncating="post", padding="post")
    attention_masks = []
    for seq in input_ids:
        seq_mask = [float(i > 0) for i in seq]
        attention_masks.append(seq_mask)
    return labels, input_ids, attention_masks, note_ids, times


def reorder_by_time(data):
    data.chartdate = pd.to_datetime(data.chartdate)
    data.charttime = pd.to_datetime(data.charttime)
    data.loc[data.charttime.isna(), 'charttime'] = data[data.charttime.isna()].chartdate + pd.Timedelta(hours=23,
                                                                                                        minutes=59,
                                                                                                        seconds=59)
    data = data.sort_values(by=['Adm_ID', 'charttime', 'Note_ID'])
    data.reset_index(inplace=True)
    return data


def concat_by_id_list(df, labels, inputs, masks, str_len):
    final_labels, final_inputs, final_masks = [], [], []
    id_lists = df.Adm_ID.unique()
    for id in id_lists:
        id_ix = df.index[df.Adm_ID == id].to_list()
        final_inputs.append(inputs[id_ix])
        final_masks.append(masks[id_ix])
        final_labels.append(labels[id_ix].max())
    return final_labels, final_inputs, final_masks, id_lists


def concat_by_id_list_with_note_chunk_id(df, labels, inputs, masks, note_ids, str_len):
    final_labels, final_inputs, final_masks, final_note_ids, final_chunk_ids = [], [], [], [], []
    id_lists = df.Adm_ID.unique()
    for id in id_lists:
        id_ix = df.index[df.Adm_ID == id].to_list()
        final_inputs.append(inputs[id_ix])
        final_masks.append(masks[id_ix])
        final_labels.append(labels[id_ix].max())
        final_note_ids.append(note_ids[id_ix])
        final_chunk_ids.append(torch.tensor(list(range(len(id_ix)))[::-1]))
    return final_labels, final_inputs, final_masks, id_lists, final_note_ids, final_chunk_ids


def concat_by_id_list_with_note_chunk_id_time(df, labels, inputs, masks, note_ids, times, str_len):
    final_labels, final_inputs, final_masks, final_note_ids, final_chunk_ids, final_times = [], [], [], [], [], []
    id_lists = df.Adm_ID.unique()
    for id in id_lists:
        id_ix = df.index[df.Adm_ID == id].to_list()
        final_inputs.append(inputs[id_ix])
        final_masks.append(masks[id_ix])
        final_labels.append(labels[id_ix].max())
        final_note_ids.append(note_ids[id_ix])
        final_chunk_ids.append(torch.tensor(list(range(len(id_ix)))[::-1]))
        final_times.append(torch.tensor(np.concatenate([np.zeros(1), np.diff(times[id_ix])])))
    return final_labels, final_inputs, final_masks, id_lists, final_note_ids, final_chunk_ids, final_times


def convert_note_ids(note_ids):
    new_dict = dict(zip(pd.Series(note_ids).unique(), range(len(pd.Series(note_ids).unique()))[::-1]))
    new_ids = [new_dict[i] for i in note_ids]
    return torch.tensor(new_ids)


def flat_accuracy(preds, labels):
    pred_flat = np.asarray([1 if i else 0 for i in (preds.flatten() >= 0.5)])
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)


def model_auc(y_true, y_pred):
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    auc_score = auc(fpr, tpr)
    return auc_score, fpr, tpr, thresholds


def model_aupr(y_true, y_pred):
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
    aupr_score = auc(recall, precision)
    return aupr_score, precision, recall, thresholds


def write_performance(flat_true_labels, flat_predictions, flat_logits, config, args):
    test_accuracy = accuracy_score(flat_true_labels, flat_predictions)

    test_f1 = f1_score(flat_true_labels, flat_predictions, average='binary')

    test_prec = precision_score(flat_true_labels, flat_predictions, average='binary')

    test_rec = recall_score(flat_true_labels, flat_predictions, average='binary')

    test_auc, _, _, _ = model_auc(flat_true_labels, flat_logits)

    test_mc = matthews_corrcoef(flat_true_labels, flat_predictions)

    test_aupr, _, _, _ = model_aupr(flat_true_labels, flat_logits)

    test_msl = args.max_seq_length

    test_seed = args.seed

    test_dir_code = args.data_dir.split('_')[-1]

    test_time = time.ctime()

    exp_path = "{}_{}_{}.csv".format(config.task_name, config.embed_mode, test_msl)

    header = "Len,Dir,Seed,Accuracy,F1_Score,Precision,Recall,AUC,MCC,AUPR,Time"
    content = "{},{},{},{},{},{},{},{},{},{},{}".format(test_msl,
                                                        test_dir_code,
                                                        test_seed,
                                                        test_accuracy,
                                                        test_f1,
                                                        test_prec,
                                                        test_rec,
                                                        test_auc,
                                                        test_mc,
                                                        test_aupr,
                                                        test_time)

    if os.path.exists(exp_path):
        with open(exp_path, 'a') as f:
            f.write(content + "\n")
    else:
        with open(exp_path, 'w') as f:
            f.write(header + "\n")
            f.write(content + "\n")

    write_log("Test Patient Level Accuracy: {}\n"
              "Test Patient Level F1 Score: {}\n"
              "Test Patient Level Precision: {}\n"
              "Test Patient Level Recall: {}\n"
              "Test Patient Level AUC: {} \n"
              "Test Patient Level Matthew's correlation coefficient: {}\n"
              "Test Patient Level AUPR: {} \n"
              "All Finished!".format(test_accuracy,
                                     test_f1,
                                     test_prec,
                                     test_rec,
                                     test_auc,
                                     test_mc,
                                     test_aupr), args.log_path)

In [ ]:
# preprocessing.py
from tqdm import tqdm, trange
import pandas as pd
import io
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import re
import argparse
from sklearn.model_selection import KFold


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--original_data",
                        default=None,
                        type=str,
                        required=True,
                        help="The input data file path."
                             " Should be the .tsv file (or other data file) for the task.")
    parser.add_argument("--output_dir",
                        default=None,
                        type=str,
                        required=True,
                        help="The output directory where the processed data will be written.")
    parser.add_argument("--temp_dir",
                        default=None,
                        type=str,
                        required=True,
                        help="The output directory where the intermediate processed data will be written.")
    parser.add_argument("--task_name",
                        default=None,
                        type=str,
                        required=True,
                        help="The name of the task.")
    parser.add_argument("--log_path",
                        default=None,
                        type=str,
                        required=True,
                        help="The log file path.")
    parser.add_argument("--id_num_neg",
                        default=None,
                        type=int,
                        required=True,
                        help="The number of admission ids that we want to use for negative category.")
    parser.add_argument("--id_num_pos",
                        default=None,
                        type=int,
                        required=True,
                        help="The number of admission ids that we want to use for positive category.")
    parser.add_argument("--random_seed",
                        default=1,
                        type=int,
                        required=True,
                        help="The random_seed for train/val/test split.")
    parser.add_argument("--bert_model",
                        default="bert-base-uncased",
                        type=str,
                        required=True,
                        help="Bert pre-trained model selected in the list: bert-base-uncased, "
                             "bert-large-uncased, bert-base-cased, bert-base-multilingual, bert-base-chinese.")

    ## Other parameters
    parser.add_argument("--Kfold",
                        default=None,
                        type=int,
                        required=False,
                        help="The number of folds that we want ot use for cross validation. "
                             "Default is not doing cross validation")

    args = parser.parse_args()
    RANDOM_SEED = args.random_seed
    LOG_PATH = args.log_path
    TEMP_DIR = args.temp_dir

    if os.path.exists(TEMP_DIR) and os.listdir(TEMP_DIR):
        raise ValueError("Temp Output directory ({}) already exists and is not empty.".format(TEMP_DIR))
    os.makedirs(TEMP_DIR, exist_ok=True)

    if os.path.exists(args.output_dir) and os.listdir(args.output_dir):
        raise ValueError("Output directory ({}) already exists and is not empty.".format(args.output_dir))
    os.makedirs(args.output_dir, exist_ok=True)

    original_df = pd.read_csv(args.original_data, header=None)
    original_df.rename(columns={0: "Adm_ID",
                                1: "Note_ID",
                                2: "chartdate",
                                3: "charttime",
                                4: "TEXT",
                                5: "Label"}, inplace=True)

    tokenizer = BertTokenizer.from_pretrained(args.bert_model, do_lower_case=True)

    write_log(("New Pre-processing Job Start! \n"
               "original_data: {}, output_dir: {}, temp_dir: {} \n"
               "task_name: {}, log_path: {}\n"
               "id_num_neg: {}, id_num_pos: {}\n"
               "random_seed: {}, bert_model: {}").format(args.original_data, args.output_dir, args.temp_dir,
                                                         args.task_name, args.log_path,
                                                         args.id_num_neg, args.id_num_pos,
                                                         args.random_seed, args.bert_model), LOG_PATH)

    for i in range(int(np.ceil(len(original_df) / 10000))):
        write_log("chunk {} tokenize start!".format(i), LOG_PATH)
        df_chunk = original_df.iloc[i * 10000:(i + 1) * 10000].copy()
        df_processed_chunk = preprocessing(df_chunk, tokenizer)
        df_processed_chunk = df_processed_chunk.astype({'Adm_ID': 'int64', 'Note_ID': 'int64', 'Label': 'int64'})
        temp_file_dir = os.path.join(TEMP_DIR, 'Processed_{}.csv'.format(i))
        df_processed_chunk.to_csv(temp_file_dir, index=False)

    df = pd.DataFrame({'Adm_ID': [], 'Note_ID': [], 'TEXT': [], 'Input_ID': [],
                       'Label': [], 'chartdate': [], 'charttime': []})
    for i in range(int(np.ceil(len(original_df) / 10000))):
        temp_file_dir = os.path.join(TEMP_DIR, 'Processed_{}.csv'.format(i))
        df_chunk = pd.read_csv(temp_file_dir, header=0)
        write_log("chunk {} has {} notes".format(i, len(df_chunk)), LOG_PATH)
        df = df.append(df_chunk, ignore_index=True)

    result = df.Label.value_counts()
    write_log(
        "In the full dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}".format(result[1],
                                                                                          result[0]),
        LOG_PATH)

    dead_ID = pd.Series(df[df.Label == 1].Adm_ID.unique())
    not_dead_ID = pd.Series(df[df.Label == 0].Adm_ID.unique())
    write_log("Total Positive Patients' ids: {}, Total Negative Patients' ids: {}".format(len(dead_ID), len(not_dead_ID)), LOG_PATH)

    not_dead_ID_use = not_dead_ID.sample(n=args.id_num_neg, random_state=RANDOM_SEED)
    dead_ID_use = dead_ID.sample(n=args.id_num_pos, random_state=RANDOM_SEED)

    if args.Kfold is None:
        id_val_test_t = dead_ID_use.sample(frac=0.2, random_state=RANDOM_SEED)
        id_val_test_f = not_dead_ID_use.sample(frac=0.2, random_state=RANDOM_SEED)

        id_train_t = dead_ID_use.drop(id_val_test_t.index)
        id_train_f = not_dead_ID_use.drop(id_val_test_f.index)

        id_val_t = id_val_test_t.sample(frac=0.5, random_state=RANDOM_SEED)
        id_test_t = id_val_test_t.drop(id_val_t.index)
        id_val_f = id_val_test_f.sample(frac=0.5, random_state=RANDOM_SEED)
        id_test_f = id_val_test_f.drop(id_val_f.index)

        id_test = pd.concat([id_test_t, id_test_f])
        test_id_label = pd.DataFrame(data=list(zip(id_test, [1] * len(id_test_t) + [0] * len(id_test_f))),
                                     columns=['id', 'label'])

        id_val = pd.concat([id_val_t, id_val_f])
        val_id_label = pd.DataFrame(data=list(zip(id_val, [1] * len(id_val_t) + [0] * len(id_val_f))),
                                    columns=['id', 'label'])

        id_train = pd.concat([id_train_t, id_train_f])
        train_id_label = pd.DataFrame(data=list(zip(id_train, [1] * len(id_train_t) + [0] * len(id_train_f))),
                                      columns=['id', 'label'])

        mortality_train = df[df.Adm_ID.isin(train_id_label.id)]
        mortality_val = df[df.Adm_ID.isin(val_id_label.id)]
        mortality_test = df[df.Adm_ID.isin(test_id_label.id)]
        mortality_not_use = df[
            (~df.Adm_ID.isin(train_id_label.id)) & (~df.Adm_ID.isin(val_id_label.id) & (~df.Adm_ID.isin(test_id_label.id)))]

        train_result = mortality_train.Label.value_counts()

        val_result = mortality_val.Label.value_counts()

        test_result = mortality_test.Label.value_counts()

        no_result = mortality_not_use.Label.value_counts()

        mortality_train.to_csv(os.path.join(args.output_dir, 'train.csv'), index=False)
        mortality_val.to_csv(os.path.join(args.output_dir, 'val.csv'), index=False)
        mortality_test.to_csv(os.path.join(args.output_dir, 'test.csv'), index=False)
        mortality_not_use.to_csv(os.path.join(args.output_dir, 'not_use.csv'), index=False)
        df.to_csv(os.path.join(args.output_dir, 'full.csv'), index=False)

        if len(no_result) == 2:
            write_log(("In the train dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                       "In the validation dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                       "In the test dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                       "In the not use dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}").format(
                train_result[1],
                train_result[0],
                val_result[1],
                val_result[0],
                test_result[1],
                test_result[0],
                no_result[1],
                no_result[0]),
                LOG_PATH)
        else:
            try:
                write_log(("In the train dataset Positive Patients' Notes: {}, Negative  Patients' Notes: {}\n"
                           "In the validation dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                           "In the test dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                           "In the not use dataset Negative Patients' Notes: {}").format(train_result[1],
                                                                                          train_result[0],
                                                                                          val_result[1],
                                                                                          val_result[0],
                                                                                          test_result[1],
                                                                                          test_result[0],
                                                                                          no_result[0]),
                          LOG_PATH)
            except KeyError:
                write_log(("In the train dataset Positive Patients' Notes: {}, Negative  Patients' Notes: {}\n"
                           "In the validation dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                           "In the test dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                           "In the not use dataset Positive Patients' Notes: {}").format(train_result[1],
                                                                                          train_result[0],
                                                                                          val_result[1],
                                                                                          val_result[0],
                                                                                          test_result[1],
                                                                                          test_result[0],
                                                                                          no_result[1]),
                          LOG_PATH)

        write_log("Data saved in the {}".format(args.output_dir), LOG_PATH)
    else:
        folds_t = KFold(args.Kfold, False, RANDOM_SEED)
        folds_f = KFold(args.Kfold, False, RANDOM_SEED)
        dead_ID_use.reset_index(inplace=True, drop=True)
        not_dead_ID_use.reset_index(inplace=True, drop=True)
        for num, ((train_t, test_t), (train_f, test_f)) in enumerate(zip(folds_t.split(dead_ID_use),
                                                                         folds_f.split(not_dead_ID_use))):
            id_train_t = dead_ID_use[train_t]
            id_val_test_t = dead_ID_use[test_t]
            id_train_f = not_dead_ID_use[train_f]
            id_val_test_f = not_dead_ID_use[test_f]
            id_val_t = id_val_test_t.sample(frac=0.5, random_state=RANDOM_SEED)
            id_test_t = id_val_test_t.drop(id_val_t.index)
            id_val_f = id_val_test_f.sample(frac=0.5, random_state=RANDOM_SEED)
            id_test_f = id_val_test_f.drop(id_val_f.index)

            id_test = pd.concat([id_test_t, id_test_f])
            test_id_label = pd.DataFrame(data=list(zip(id_test, [1] * len(id_test_t) + [0] * len(id_test_f))),
                                         columns=['id', 'label'])

            id_val = pd.concat([id_val_t, id_val_f])
            val_id_label = pd.DataFrame(data=list(zip(id_val, [1] * len(id_val_t) + [0] * len(id_val_f))),
                                        columns=['id', 'label'])

            id_train = pd.concat([id_train_t, id_train_f])
            train_id_label = pd.DataFrame(data=list(zip(id_train, [1] * len(id_train_t) + [0] * len(id_train_f))),
                                          columns=['id', 'label'])

            mortality_train = df[df.Adm_ID.isin(train_id_label.id)]
            mortality_val = df[df.Adm_ID.isin(val_id_label.id)]
            mortality_test = df[df.Adm_ID.isin(test_id_label.id)]
            mortality_not_use = df[
                (~df.Adm_ID.isin(train_id_label.id)) & (
                            ~df.Adm_ID.isin(val_id_label.id) & (~df.Adm_ID.isin(test_id_label.id)))]

            train_result = mortality_train.Label.value_counts()

            val_result = mortality_val.Label.value_counts()

            test_result = mortality_test.Label.value_counts()

            no_result = mortality_not_use.Label.value_counts()

            os.makedirs(os.path.join(args.output_dir, str(num)))
            mortality_train.to_csv(os.path.join(args.output_dir, str(num), 'train.csv'), index=False)
            mortality_val.to_csv(os.path.join(args.output_dir, str(num), 'val.csv'), index=False)
            mortality_test.to_csv(os.path.join(args.output_dir, str(num), 'test.csv'), index=False)
            mortality_not_use.to_csv(os.path.join(args.output_dir, str(num), 'not_use.csv'), index=False)
            df.to_csv(os.path.join(args.output_dir, str(num), 'full.csv'), index=False)

            if len(no_result) == 2:
                write_log(("In the {}th split of {} folds\n"
                           "In the train dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                           "In the validation dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                           "In the test dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                           "In the not use dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}").format(
                    num,
                    args.Kfold,
                    train_result[1],
                    train_result[0],
                    val_result[1],
                    val_result[0],
                    test_result[1],
                    test_result[0],
                    no_result[1],
                    no_result[0]),
                    LOG_PATH)
            else:
                try:
                    write_log(("In the {}th split of {} folds\n"
                               "In the train dataset Positive Patients' Notes: {}, Negative  Patients' Notes: {}\n"
                               "In the validation dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                               "In the test dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                               "In the not use dataset Negative Patients' Notes: {}").format(num,
                                                                                              args.Kfold,
                                                                                              train_result[1],
                                                                                              train_result[0],
                                                                                              val_result[1],
                                                                                              val_result[0],
                                                                                              test_result[1],
                                                                                              test_result[0],
                                                                                              no_result[0]),
                              LOG_PATH)
                except KeyError:
                    write_log(("In the {}th split of {} folds\n"
                               "In the train dataset Positive Patients' Notes: {}, Negative  Patients' Notes: {}\n"
                               "In the validation dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                               "In the test dataset Positive Patients' Notes: {}, Negative Patients' Notes: {}\n"
                               "In the not use dataset Positive Patients' Notes: {}").format(num,
                                                                                             args.Kfold,
                                                                                             train_result[1],
                                                                                             train_result[0],
                                                                                             val_result[1],
                                                                                             val_result[0],
                                                                                             test_result[1],
                                                                                             test_result[0],
                                                                                             no_result[1]),
                              LOG_PATH)

            write_log("Data saved in the {}".format(os.path.join(args.output_dir, str(num))), LOG_PATH)


if __name__ == "__main__":
    main()

In [ ]:
# split_into_chuck.py
import pandas as pd
from tqdm import trange, tqdm
import argparse
import os


def main():
    parser = argparse.ArgumentParser()
    ## Required parameters
    parser.add_argument("--data_dir",
                        default=None,
                        type=str,
                        required=True,
                        help="The input data dir. Should contain the .tsv files (or other data files) for the task.")

    parser.add_argument("--train_data",
                        default=None,
                        type=str,
                        required=True,
                        help="The input training data file name."
                             " Should be the .tsv file (or other data file) for the task.")

    parser.add_argument("--val_data",
                        default=None,
                        type=str,
                        required=True,
                        help="The input validation data file name."
                             " Should be the .tsv file (or other data file) for the task.")

    parser.add_argument("--test_data",
                        default=None,
                        type=str,
                        required=True,
                        help="The input test data file name."
                             " Should be the .tsv file (or other data file) for the task.")

    parser.add_argument("--log_path",
                        default=None,
                        type=str,
                        required=True,
                        help="The log file path.")

    parser.add_argument("--output_dir",
                        default=None,
                        type=str,
                        required=True,
                        help="The output directory where the model checkpoints will be written.")

    ## Other parameters
    parser.add_argument("--max_seq_length",
                        default=128,
                        type=int,
                        help="The maximum total input sequence length after WordPiece tokenization. \n"
                             "Sequences longer than this will be truncated, and sequences shorter \n"
                             "than this will be padded.")

    args = parser.parse_args()
    if os.path.exists(args.output_dir) and os.listdir(args.output_dir):
        raise ValueError("Output directory ({}) already exists and is not empty.".format(args.output_dir))
    os.makedirs(args.output_dir, exist_ok=True)

    LOG_PATH = args.log_path
    MAX_LEN = args.max_seq_length

    write_log(("New Split Job Start! \n"
               "data_dir: {}, train_data: {}, val_data: {}, test_data: {} \n"
               "log_path: {}, output_dir: {}, max_seq_length: {}").format(args.data_dir, args.train_data,
                                                                        args.val_data, args.test_data,
                                                                        args.log_path, args.output_dir,
                                                                        args.max_seq_length), LOG_PATH)

    train_file_path = os.path.join(args.data_dir, args.train_data)
    val_file_path = os.path.join(args.data_dir, args.val_data)
    test_file_path = os.path.join(args.data_dir, args.test_data)
    train_df = pd.read_csv(train_file_path)
    val_df = pd.read_csv(val_file_path)
    test_df = pd.read_csv(test_file_path)

    new_train_df = split_into_chunks(train_df, MAX_LEN)
    new_val_df = split_into_chunks(val_df, MAX_LEN)
    new_test_df = split_into_chunks(test_df, MAX_LEN)

    train_result = new_train_df.Label.value_counts()
    val_result = new_val_df.Label.value_counts()
    test_result = new_test_df.Label.value_counts()

    write_log(("In the train dataset Positive Patients' Chunks: {}, Negative Patients' Chunks: {}\n"
               "In the validation dataset Positive Patients' Chunks: {}, Negative Patients' Chunks: {}\n"
               "In the test dataset Positive Patients' Chunks: {}, Negative Patients' Chunks: {}").format(train_result[1],
                                                                                                  train_result[0],
                                                                                                  val_result[1],
                                                                                                  val_result[0],
                                                                                                  test_result[1],
                                                                                                  test_result[0]),
              LOG_PATH)

    new_train_df.to_csv(os.path.join(args.output_dir, args.train_data), index=False)
    new_val_df.to_csv(os.path.join(args.output_dir, args.val_data), index=False)
    new_test_df.to_csv(os.path.join(args.output_dir, args.test_data), index=False)

    write_log("Split finished", LOG_PATH)


if __name__ == "__main__":
    main()

In [ ]:
import time
import os
import torch
import random
import argparse
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
from pytorch_transformers import BertTokenizer, BertConfig
from pytorch_pretrained_bert.optimization import BertAdam
from modeling_readmission import BertModel
from modeling_patient import FTLSTMLayer
from tqdm import tqdm, trange
import pandas as pd
import io
import numpy as np
import matplotlib.pyplot as plt
from dotmap import DotMap
from torch import nn
import re
from other_func import write_log, Tokenize_with_note_id_hour, concat_by_id_list_with_note_chunk_id_time
from other_func import convert_note_ids, flat_accuracy, write_performance, reorder_by_time
from utils import time_batch_generator


def main():
    parser = argparse.ArgumentParser()
    ## Required parameters
    parser.add_argument("--data_dir",
                        default=None,
                        type=str,
                        required=True,
                        help="The input data dir. Should contain the .tsv files (or other data files) for the task.")

    parser.add_argument("--train_data",
                        default=None,
                        type=str,
                        required=True,
                        help="The input training data file name."
                             " Should be the .tsv file (or other data file) for the task.")

    parser.add_argument("--val_data",
                        default=None,
                        type=str,
                        required=True,
                        help="The input validation data file name."
                             " Should be the .tsv file (or other data file) for the task.")

    parser.add_argument("--test_data",
                        default=None,
                        type=str,
                        required=True,
                        help="The input test data file name."
                             " Should be the .tsv file (or other data file) for the task.")

    parser.add_argument("--log_path",
                        default=None,
                        type=str,
                        required=True,
                        help="The log file path.")

    parser.add_argument("--output_dir",
                        default=None,
                        type=str,
                        required=True,
                        help="The output directory where the model checkpoints will be written.")

    parser.add_argument("--save_model",
                        default=False,
                        action='store_true',
                        help="Whether to save the model.")

    parser.add_argument("--bert_model",
                        default="bert-base-uncased",
                        type=str,
                        required=True,
                        help="Bert pre-trained model selected in the list: bert-base-uncased, "
                             "bert-large-uncased, bert-base-cased, bert-base-multilingual, bert-base-chinese.")

    parser.add_argument("--embed_mode",
                        default=None,
                        type=str,
                        required=True,
                        help="The embedding type selected in the list: all, note, chunk, no.")

    parser.add_argument("--task_name",
                        default="FTLSTM_with_ClBERT_mortality",
                        type=str,
                        required=True,
                        help="The name of the task.")

    ## Other parameters
    parser.add_argument("--max_seq_length",
                        default=128,
                        type=int,
                        help="The maximum total input sequence length after WordPiece tokenization. \n"
                             "Sequences longer than this will be truncated, and sequences shorter \n"
                             "than this will be padded.")
    parser.add_argument("--max_chunk_num",
                        default=64,
                        type=int,
                        help="The maximum total input chunk numbers after WordPiece tokenization.")
    parser.add_argument("--train_batch_size",
                        default=1,
                        type=int,
                        help="Total batch size for training.")
    parser.add_argument("--eval_batch_size",
                        default=1,
                        type=int,
                        help="Total batch size for eval.")
    parser.add_argument("--learning_rate",
                        default=2e-5,
                        type=float,
                        help="The initial learning rate for Adam.")
    parser.add_argument("--warmup_proportion",
                        default=0.0,
                        type=float,
                        help="Proportion of training to perform linear learning rate warmup for. "
                             "E.g., 0.1 = 10%% of training.")
    parser.add_argument("--num_train_epochs",
                        default=3,
                        type=int,
                        help="Total number of training epochs to perform.")
    parser.add_argument('--seed',
                        type=int,
                        default=42,
                        help="random seed for initialization")
    parser.add_argument('--gradient_accumulation_steps',
                        type=int,
                        default=1,
                        help="Number of updates steps to accumualte before performing a backward/update pass.")

    args = parser.parse_args()

    if os.path.exists(args.output_dir) and os.listdir(args.output_dir) and args.save_model:
        raise ValueError("Output directory ({}) already exists and is not empty.".format(args.output_dir))
    os.makedirs(args.output_dir, exist_ok=True)

    LOG_PATH = args.log_path
    MAX_LEN = args.max_seq_length

    config = DotMap()
    config.hidden_dropout_prob = 0.1
    config.layer_norm_eps = 1e-12
    config.initializer_range = 0.02
    config.max_note_position_embedding = 1000
    config.max_chunk_position_embedding = 1000
    config.embed_mode = args.embed_mode
    config.layer_norm_eps = 1e-12
    config.hidden_size = 768
    config.lstm_layers = 1

    config.task_name = args.task_name

    write_log(("New Job Start! \n"
               "Data directory: {}, Directory Code: {}, Save Model: {}\n"
               "Output_dir: {}, Task Name: {}, embed_mode: {}\n"
               "max_seq_length: {},  max_chunk_num: {}\n"
               "train_batch_size: {}, eval_batch_size: {}\n"
               "learning_rate: {}, warmup_proportion: {}\n"
               "num_train_epochs: {}, seed: {}, gradient_accumulation_steps: {}\n"
               "FTLSTM Model's lstm_layers: {}").format(args.data_dir,
                                                        args.data_dir.split('_')[-1],
                                                        args.save_model,
                                                        args.output_dir,
                                                        config.task_name,
                                                        config.embed_mode,
                                                        args.max_seq_length,
                                                        args.max_chunk_num,
                                                        args.train_batch_size,
                                                        args.eval_batch_size,
                                                        args.learning_rate,
                                                        args.warmup_proportion,
                                                        args.num_train_epochs,
                                                        args.seed,
                                                        args.gradient_accumulation_steps,
                                                        config.lstm_layers),
              LOG_PATH)

    content = "config setting: \n"
    for k, v in config.items():
        content += "{}: {} \n".format(k, v)
    write_log(content, LOG_PATH)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    n_gpu = torch.cuda.device_count()
    write_log("Number of GPU is {}".format(n_gpu), LOG_PATH)
    for i in range(n_gpu):
        write_log(("Device Name: {},"
                   "Device Capability: {}").format(torch.cuda.get_device_name(i),
                                                   torch.cuda.get_device_capability(i)), LOG_PATH)

    train_file_path = os.path.join(args.data_dir, args.train_data)
    val_file_path = os.path.join(args.data_dir, args.val_data)
    test_file_path = os.path.join(args.data_dir, args.test_data)
    train_df = pd.read_csv(train_file_path)
    val_df = pd.read_csv(val_file_path)
    test_df = pd.read_csv(test_file_path)

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)
    if n_gpu > 0:
        torch.cuda.manual_seed_all(args.seed)

    tokenizer = BertTokenizer.from_pretrained(args.bert_model, do_lower_case=True)

    write_log("Tokenize Start!", LOG_PATH)
    train_df = reorder_by_time(train_df)
    val_df = reorder_by_time(val_df)
    test_df = reorder_by_time(test_df)
    train_labels, train_inputs, train_masks, train_note_ids, train_times = Tokenize_with_note_id_hour(train_df, MAX_LEN,
                                                                                                      tokenizer)
    validation_labels, validation_inputs, validation_masks, validation_note_ids, validation_times = Tokenize_with_note_id_hour(
        val_df, MAX_LEN, tokenizer)
    test_labels, test_inputs, test_masks, test_note_ids, test_times = Tokenize_with_note_id_hour(test_df, MAX_LEN,
                                                                                                 tokenizer)
    write_log("Tokenize Finished!", LOG_PATH)
    train_inputs = torch.tensor(train_inputs)
    validation_inputs = torch.tensor(validation_inputs)
    test_inputs = torch.tensor(test_inputs)
    train_labels = torch.tensor(train_labels)
    validation_labels = torch.tensor(validation_labels)
    test_labels = torch.tensor(test_labels)
    train_masks = torch.tensor(train_masks)
    validation_masks = torch.tensor(validation_masks)
    test_masks = torch.tensor(test_masks)
    train_times = torch.tensor(train_times)
    validation_times = torch.tensor(validation_times)
    test_times = torch.tensor(test_times)
    write_log(("train dataset size is %d,\n"
               "validation dataset size is %d,\n"
               "test dataset size is %d") % (len(train_inputs), len(validation_inputs), len(test_inputs)), LOG_PATH)

    (train_labels, train_inputs,
     train_masks, train_ids,
     train_note_ids, train_chunk_ids, train_times) = concat_by_id_list_with_note_chunk_id_time(train_df, train_labels,
                                                                                               train_inputs,
                                                                                               train_masks,
                                                                                               train_note_ids,
                                                                                               train_times, MAX_LEN)
    (validation_labels, validation_inputs,
     validation_masks, validation_ids,
     validation_note_ids, validation_chunk_ids,
     validation_times) = concat_by_id_list_with_note_chunk_id_time(val_df, validation_labels,
                                                                   validation_inputs, validation_masks,
                                                                   validation_note_ids, validation_times,
                                                                   MAX_LEN)
    (test_labels, test_inputs,
     test_masks, test_ids,
     test_note_ids, test_chunk_ids, test_times) = concat_by_id_list_with_note_chunk_id_time(test_df, test_labels,
                                                                                            test_inputs, test_masks,
                                                                                            test_note_ids, test_times,
                                                                                            MAX_LEN)

    model = BertModel.from_pretrained(args.bert_model).to(device)
    model.to(device)
    lstm_layer = FTLSTMLayer(config=config, num_labels=1)
    lstm_layer.to(device)

    if n_gpu > 1:
        model = torch.nn.DataParallel(model)
        lstm_layer = torch.nn.DataParallel(lstm_layer)
    param_optimizer = list(model.named_parameters()) + list(lstm_layer.named_parameters())
    no_decay = ['bias', 'gamma', 'beta']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in param_optimizer if not any(nd in n for nd in no_decay)],
         'weight_decay_rate': 0.01},
        {'params': [p for n, p in param_optimizer if any(nd in n for nd in no_decay)],
         'weight_decay_rate': 0.0}
    ]

    num_train_steps = int(
        len(train_labels) / args.gradient_accumulation_steps * args.num_train_epochs)

    optimizer = BertAdam(optimizer_grouped_parameters,
                         lr=args.learning_rate,
                         warmup=args.warmup_proportion,
                         t_total=num_train_steps)
    start = time.time()
    # Store our loss and accuracy for plotting
    train_loss_set = []

    # Number of training epochs (authors recommend between 2 and 4)
    epochs = args.num_train_epochs

    train_batch_generator = time_batch_generator(args.max_chunk_num, train_inputs, train_labels, train_masks,
                                                 train_note_ids, train_chunk_ids, train_times)
    validation_batch_generator = time_batch_generator(args.max_chunk_num, validation_inputs, validation_labels,
                                                      validation_masks, validation_note_ids, validation_chunk_ids,
                                                      validation_times)

    write_log("Training start!", LOG_PATH)
    # trange is a tqdm wrapper around the normal python range
    with torch.autograd.set_detect_anomaly(False):
        for epoch in trange(epochs, desc="Epoch"):

            # Training

            # Set our model to training mode (as opposed to evaluation mode)
            model.train()
            lstm_layer.train()

            # Tracking variables
            tr_loss = 0
            nb_tr_examples, nb_tr_steps = 0, 0

            # Train the data for one epoch
            tr_ids_num = len(train_ids)
            tr_batch_loss = []
            for step in range(tr_ids_num):
                b_input_ids, b_labels, b_input_mask, b_note_ids, b_chunk_ids, b_times = next(train_batch_generator)
                b_input_ids = b_input_ids.to(device)
                b_input_mask = b_input_mask.to(device)
                b_new_note_ids = convert_note_ids(b_note_ids).to(device)
                b_chunk_ids = b_chunk_ids.unsqueeze(0).to(device)
                b_labels = b_labels.to(device)
                b_labels.resize_((1))
                _, whole_output = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
                whole_input = whole_output.unsqueeze(0)
                b_new_note_ids = b_new_note_ids.unsqueeze(0)
                b_times = b_times.unsqueeze(0).to(device)
                loss, pred = lstm_layer(whole_input, b_times, b_new_note_ids, b_chunk_ids, b_labels)

                if n_gpu > 1:
                    loss = loss.mean()  # mean() to average on multi-gpu.
                tr_batch_loss.append(loss.item())

                # Backward pass
                loss.backward()
                # Update parameters and take a step using the computed gradient
                if (step + 1) % args.train_batch_size == 0:
                    optimizer.step()
                    optimizer.zero_grad()
                    train_loss_set.append(np.mean(tr_batch_loss))
                    tr_batch_loss = []
                # Update tracking variables
                tr_loss += loss.item()
                nb_tr_examples += b_input_ids.size(0)
                nb_tr_steps += 1

            write_log("Train loss: {}".format(tr_loss / nb_tr_steps), LOG_PATH)
            # Validation

            # Put model in evaluation mode to evaluate loss on the validation set
            model.eval()
            lstm_layer.eval()

            # Tracking variables
            eval_loss, eval_accuracy = 0, 0
            nb_eval_steps, nb_eval_examples = 0, 0
            # Evaluate data for one epoch
            ev_ids_num = len(validation_ids)
            for step in range(ev_ids_num):
                with torch.no_grad():
                    b_input_ids, b_labels, b_input_mask, b_note_ids, b_chunk_ids, b_times = next(
                        validation_batch_generator)
                    b_input_ids = b_input_ids.to(device)
                    b_input_mask = b_input_mask.to(device)
                    b_new_note_ids = convert_note_ids(b_note_ids).to(device)
                    b_chunk_ids = b_chunk_ids.unsqueeze(0).to(device)
                    b_labels.resize_((1))
                    _, whole_output = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
                    whole_input = whole_output.unsqueeze(0)
                    b_new_note_ids = b_new_note_ids.unsqueeze(0)
                    b_times = b_times.unsqueeze(0).to(device)
                    pred = lstm_layer(whole_input, b_times, b_new_note_ids, b_chunk_ids).detach().cpu().numpy()
                label_ids = b_labels.numpy()
                tmp_eval_accuracy = flat_accuracy(pred, label_ids)
                eval_accuracy += tmp_eval_accuracy
                nb_eval_steps += 1

            write_log("Validation Accuracy: {}".format(eval_accuracy / nb_eval_steps), LOG_PATH)
            output_checkpoints_path = os.path.join(args.output_dir,
                                                   "bert_fine_tuned_with_note_checkpoint_%d.pt" % epoch)

            if args.save_model:
                if n_gpu > 1:
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.module.state_dict(),
                        'lstm_layer_state_dict': lstm_layer.module.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': loss,
                    },
                        output_checkpoints_path)
                else:
                    torch.save({
                        'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'lstm_layer_state_dict': lstm_layer.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'loss': loss,
                    },
                        output_checkpoints_path)
    end = time.time()
    write_log("total training time is: {}s".format(end - start), LOG_PATH)

    fig1 = plt.figure(figsize=(15, 8))
    plt.title("Training loss")
    plt.xlabel("Patient Batch")
    plt.ylabel("Loss")
    plt.plot(train_loss_set)
    if args.save_model:
        output_fig_path = os.path.join(args.output_dir, "bert_fine_tuned_with_note_training_loss.png")
        plt.savefig(output_fig_path, dpi=fig1.dpi)
        output_model_state_dict_path = os.path.join(args.output_dir, "bert_fine_tuned_with_note_state_dict.pt")
        if n_gpu > 1:
            torch.save({
                'model_state_dict': model.module.state_dict(),
                'lstm_layer_state_dict': lstm_layer.module.state_dict(),
            },
                output_model_state_dict_path)
        else:
            torch.save({
                'model_state_dict': model.state_dict(),
                'lstm_layer_state_dict': lstm_layer.state_dict(),
            },
                output_model_state_dict_path)
        write_log("Model saved!", LOG_PATH)
    else:
        output_fig_path = os.path.join(args.output_dir,
                                       "bert_fine_tuned_with_note_training_loss_{}_{}.png".format(args.seed,
                                                                                                  args.data_dir.split(
                                                                                                      '_')[-1]))
        plt.savefig(output_fig_path, dpi=fig1.dpi)
        write_log("Model not saved as required", LOG_PATH)

    # Prediction on test set

    # Put model in evaluation mode
    model.eval()
    lstm_layer.eval()

    # Tracking variables
    predictions, true_labels = [], []

    # Predict
    te_ids_num = len(test_ids)
    for step in range(te_ids_num):
        b_input_ids = test_inputs[step][-args.max_chunk_num:, :].to(device)
        b_input_mask = test_masks[step][-args.max_chunk_num:, :].to(device)
        b_note_ids = test_note_ids[step][-args.max_chunk_num:]
        b_new_note_ids = convert_note_ids(b_note_ids).to(device)
        b_chunk_ids = test_chunk_ids[step][-args.max_chunk_num:].unsqueeze(0).to(device)
        b_labels = test_labels[step]
        b_labels.resize_((1))
        with torch.no_grad():
            _, whole_output = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
            whole_input = whole_output.unsqueeze(0)
            b_new_note_ids = b_new_note_ids.unsqueeze(0)
            b_times = test_times[step][-args.max_chunk_num:].unsqueeze(0).to(device)
            pred = lstm_layer(whole_input, b_times, b_new_note_ids, b_chunk_ids).detach().cpu().numpy()
        label_ids = b_labels.numpy()[0]
        predictions.append(pred)
        true_labels.append(label_ids)

    # Flatten the predictions and true values for aggregate Matthew's evaluation on the whole dataset
    flat_logits = [item for sublist in predictions for item in sublist]
    flat_predictions = np.asarray([1 if i else 0 for i in (np.array(flat_logits) >= 0.5)])
    flat_true_labels = np.asarray(true_labels)

    output_df = pd.DataFrame({'pred_prob': flat_logits,
                              'pred_label': flat_predictions,
                              'label': flat_true_labels,
                              'Adm_ID': test_ids})

    if args.save_model:
        output_df.to_csv(os.path.join(args.output_dir, 'test_predictions.csv'), index=False)
    else:
        output_df.to_csv(os.path.join(args.output_dir,
                                      'test_predictions_{}_{}.csv'.format(args.seed,
                                                                          args.data_dir.split('_')[-1])), index=False)

    write_performance(flat_true_labels, flat_predictions, flat_logits, config, args)


if __name__ == "__main__":
    main()